# 11 — Subarray Sum Problems (Basic to Advanced)

## Learning goals
- Understand subarrays and their boundaries.
- Count, find, and maximize subarrays using prefix sums.
- Know when to use a hash map, sliding window, or Kadane's algorithm.
- Handle negative numbers, zeros, and repeated prefix sums.
- Derive an approach from the exact wording of a problem.

**Important distinction:** A subarray is contiguous. A subsequence does not need to be contiguous.

## 1. What is a subarray?

For `nums = [2, 4, 6]`, examples of subarrays include:
- `[2]`
- `[4, 6]`
- `[2, 4, 6]`

`[2, 6]` is not a subarray because the elements are not adjacent in the original array.

An array of length `n` has `n * (n + 1) // 2` non-empty subarrays.

In [1]:
def count_all_nonempty_subarrays(n: int) -> int:
    return n * (n + 1) // 2

print(count_all_nonempty_subarrays(3))  # 6

6


## 2. Three common subarray goals

Read the required output carefully:

1. **Count:** How many subarrays satisfy a condition?
2. **Find:** Return one valid subarray or its indices.
3. **Optimize:** Return the longest, shortest, maximum-sum, or minimum-sum subarray.

The same prefix-sum equation can support different goals, but the hash map must store different information.

## 3. Brute force baseline

Choose every starting index and extend the ending index while maintaining a running sum. This is `O(n^2)` time and `O(1)` extra space.

This is a useful baseline and a way to verify optimized solutions on small test cases.

In [2]:
def count_subarrays_bruteforce(nums: list[int], k: int) -> int:
    answer = 0

    for left in range(len(nums)):
        current_sum = 0

        for right in range(left, len(nums)):
            current_sum += nums[right]
            if current_sum == k:
                answer += 1

    return answer

print(count_subarrays_bruteforce([1, 2, 1, 2], 3))  # 3

3


## 4. Prefix sum equation

Define `prefix` as the sum of elements before the current position.

For a subarray sum to equal `k`:

\[
current\_prefix - previous\_prefix = k
\]

Rearrange:

\[
previous\_prefix = current\_prefix - k
\]

At each position, look up `current_prefix - k` among earlier prefix sums.

## 5. Pattern A — Count subarrays with sum k

Store the frequency of each earlier prefix sum.

- Start with `freq = {0: 1}` to represent the empty prefix.
- Add the current number to the prefix.
- Add the frequency of `prefix - k` to the answer.
- Record the current prefix.

Lookup must happen before recording the current prefix.

In [3]:
def subarray_sum_count(nums: list[int], k: int) -> int:
    prefix = 0
    answer = 0
    freq = {0: 1}

    for num in nums:
        prefix += num
        answer += freq.get(prefix - k, 0)
        freq[prefix] = freq.get(prefix, 0) + 1

    return answer

print(subarray_sum_count([1, 1, 1], 2))  # 2
print(subarray_sum_count([1, -1, 0], 0)) # 3
print(subarray_sum_count([], 0))         # 0

2
3
0


## 6. Pattern B — Find one subarray with sum k

Store the first index where each prefix sum appears.

If `prefix - k` was first seen at index `j`, the current subarray starts at `j + 1`.

Use `{0: -1}` to represent the empty prefix before index zero.

In [4]:
def find_subarray_indices(nums: list[int], k: int):
    first = {0: -1}
    prefix = 0

    for i, num in enumerate(nums):
        prefix += num
        needed = prefix - k

        if needed in first:
            return first[needed] + 1, i

        if prefix not in first:
            first[prefix] = i

    return None

print(find_subarray_indices([2, 4, -1, 3], 3))

(1, 2)


## 7. Pattern C — Longest subarray with sum k

Again store the **first** index for each prefix sum.

Why first? If the same prefix appeared earlier, using its earliest index produces the longest subarray ending at the current position.

Do not overwrite the first index.

In [5]:
def longest_subarray_sum_k(nums: list[int], k: int) -> int:
    first = {0: -1}
    prefix = 0
    best = 0

    for i, num in enumerate(nums):
        prefix += num
        needed = prefix - k

        if needed in first:
            best = max(best, i - first[needed])

        if prefix not in first:
            first[prefix] = i

    return best

print(longest_subarray_sum_k([1, -1, 5, -2, 3], 3))  # 4
print(longest_subarray_sum_k([-2, -1, 2, 1], 1))     # 2

4
2


## 8. Pattern D — Longest subarray with equal 0s and 1s

Transform each `0` into `-1` and keep each `1` as `+1`.

A subarray has equal numbers of zeroes and ones exactly when its transformed sum is zero.

Then use the longest zero-sum subarray pattern.

In [6]:
def find_max_length_equal_zeros_ones(nums: list[int]) -> int:
    first = {0: -1}
    prefix = 0
    best = 0

    for i, num in enumerate(nums):
        prefix += 1 if num == 1 else -1

        if prefix in first:
            best = max(best, i - first[prefix])
        else:
            first[prefix] = i

    return best

print(find_max_length_equal_zeros_ones([0, 1, 0]))       # 2
print(find_max_length_equal_zeros_ones([0, 1, 0, 1, 1, 0])) # 6

2
6


## 9. Pattern E — Subarray sum divisible by k

A subarray sum is divisible by `k` if its two prefix sums have the same remainder modulo `k`.

Store the frequency of each remainder. Initialize remainder zero with frequency one.

The usual problem constraints guarantee `k != 0`.

In [7]:
def count_subarrays_divisible_by_k(nums: list[int], k: int) -> int:
    freq = {0: 1}
    prefix = 0
    answer = 0

    for num in nums:
        prefix += num
        remainder = prefix % k

        answer += freq.get(remainder, 0)
        freq[remainder] = freq.get(remainder, 0) + 1

    return answer

print(count_subarrays_divisible_by_k([4, 5, 0, -2, -3, 1], 5))  # 7

7


## 10. Pattern F — Maximum subarray sum (Kadane's algorithm)

This is a different subarray pattern. We want the maximum possible sum of a non-empty contiguous subarray.

At each index, either:
- Start a new subarray at the current number, or
- Extend the previous best subarray ending at the previous index.

Transition:

\[
current = \max(num, current + num)
\]

Track the best value seen so far. Initializing with the first element handles arrays containing only negative values.

In [8]:
def max_subarray_sum(nums: list[int]) -> int:
    if not nums:
        raise ValueError("nums must be non-empty")

    current = best = nums[0]

    for num in nums[1:]:
        current = max(num, current + num)
        best = max(best, current)

    return best

print(max_subarray_sum([-2, 1, -3, 4, -1, 2, 1, -5, 4]))  # 6
print(max_subarray_sum([-5, -2, -8]))                      # -2

6
-2


## 11. Pattern G — Maximum sum with a fixed window size

If every candidate subarray must have exactly `k` elements, use a fixed-size sliding window.

1. Compute the first window sum.
2. Move one step right.
3. Add the new element and subtract the element leaving the window.

This works with negative numbers too because the window size is fixed.

In [9]:
def max_sum_fixed_window(nums: list[int], k: int) -> int:
    if k <= 0 or k > len(nums):
        raise ValueError("k must be between 1 and len(nums)")

    window_sum = sum(nums[:k])
    best = window_sum

    for right in range(k, len(nums)):
        window_sum += nums[right] - nums[right - k]
        best = max(best, window_sum)

    return best

print(max_sum_fixed_window([2, 1, 5, 1, 3, 2], 3))  # 9

9


## 12. Pattern H — Minimum-length subarray with sum at least target

When all numbers are **positive**, a variable-size sliding window works:
- Expand right until the sum reaches the target.
- While the sum is still at least target, record the length and shrink from the left.

With negative numbers, this simple shrinking rule is not valid. Advanced solutions may use prefix sums with a monotonic deque.

In [10]:
def min_subarray_len_positive(target: int, nums: list[int]) -> int:
    left = 0
    window_sum = 0
    best = float("inf")

    for right, num in enumerate(nums):
        window_sum += num

        while window_sum >= target:
            best = min(best, right - left + 1)
            window_sum -= nums[left]
            left += 1

    return 0 if best == float("inf") else best

print(min_subarray_len_positive(7, [2, 3, 1, 2, 4, 3]))  # 2

2


## 13. Choosing the correct pattern

| Problem asks for | Pattern |
|---|---|
| Number of subarrays with sum exactly k | Prefix sum + frequency map |
| One subarray with sum k | Prefix sum + seen prefix map |
| Longest subarray with sum k | Prefix sum + first-index map |
| Equal 0s and 1s, longest length | Transform + prefix sum + first index |
| Number of subarrays divisible by k | Prefix remainder + frequency map |
| Maximum sum of any non-empty subarray | Kadane's algorithm |
| Maximum sum of exactly k elements | Fixed-size sliding window |
| Minimum length with sum at least target, positive values | Variable-size sliding window |

The word “subarray” alone does not tell you which algorithm to use.

## 14. Negative numbers: the critical decision

A simple variable-size sliding window generally requires non-negative values (and often strictly positive values for particular shrinking logic).

Why? With negative numbers, extending the window can reduce its sum, and shrinking it can increase its sum. The monotonic behavior is lost.

Prefix-sum + hash-map equations still work with negative numbers because they rely on exact arithmetic equality, not monotonic movement.

## 15. Common mistakes

1. Confusing a subarray with a subsequence.
2. Forgetting `{0: 1}` for counting or `{0: -1}` for index-based patterns.
3. Using a frequency map when the task needs the earliest index.
4. Overwriting the earliest prefix index when finding the longest subarray.
5. Using sliding window with negative values without proving it is valid.
6. Initializing Kadane's algorithm to zero, which fails for all-negative arrays.
7. Forgetting fixed-window bounds when `k` is invalid.
8. Returning a length when the problem asks for a count, or vice versa.

## 16. Interview checklist

Before coding:
1. Is the required segment contiguous?
2. Is the task count, existence, one example, longest, shortest, or maximum sum?
3. Can values be negative?
4. Is the window fixed-size or variable-size?
5. If using prefix sums, should the map store counts, first indices, or remainders?
6. What initial state handles a subarray starting at index zero?

After coding, test:
- Empty input if allowed.
- One element.
- All zeroes.
- All negative values.
- Repeated prefix sums.
- A valid subarray starting at index zero.
- No valid subarray.

## 17. Practice questions — recommended order

### Easy
1. [1732. Find the Highest Altitude](https://leetcode.com/problems/find-the-highest-altitude/)
2. [724. Find Pivot Index](https://leetcode.com/problems/find-pivot-index/)
3. [53. Maximum Subarray](https://leetcode.com/problems/maximum-subarray/)

### Medium
4. [560. Subarray Sum Equals K](https://leetcode.com/problems/subarray-sum-equals-k/)
5. [525. Contiguous Array](https://leetcode.com/problems/contiguous-array/)
6. [974. Subarray Sums Divisible by K](https://leetcode.com/problems/subarray-sums-divisible-by-k/)
7. [209. Minimum Size Subarray Sum](https://leetcode.com/problems/minimum-size-subarray-sum/)
8. [325. Maximum Size Subarray Sum Equals k](https://leetcode.com/problems/maximum-size-subarray-sum-equals-k/) — Premium

### Advanced
9. [862. Shortest Subarray with Sum at Least K](https://leetcode.com/problems/shortest-subarray-with-sum-at-least-k/)
10. [152. Maximum Product Subarray](https://leetcode.com/problems/maximum-product-subarray/)

Suggested order: 1732 → 724 → 53 → 560 → 525 → 974 → 209 → 862 → 152.

For each problem, explain why your chosen pattern applies before writing code.

## 18. Reusable templates

### Count subarrays with sum k
```python
def count_sum_k(nums, k):
    freq = {0: 1}
    prefix = answer = 0

    for num in nums:
        prefix += num
        answer += freq.get(prefix - k, 0)
        freq[prefix] = freq.get(prefix, 0) + 1

    return answer
```

### Longest subarray with sum k
```python
def longest_sum_k(nums, k):
    first = {0: -1}
    prefix = best = 0

    for i, num in enumerate(nums):
        prefix += num

        if prefix - k in first:
            best = max(best, i - first[prefix - k])

        if prefix not in first:
            first[prefix] = i

    return best
```

### Kadane
```python
def max_subarray(nums):
    current = best = nums[0]

    for num in nums[1:]:
        current = max(num, current + num)
        best = max(best, current)

    return best
```

**Final mental model**
- Count → store prefix frequencies.
- Longest → store first prefix indices.
- Maximum sum → Kadane.
- Fixed length → fixed sliding window.
- Positive values and threshold → variable sliding window.
- Negative values with exact target → prefix sum + hash map.